# Graph Generation for a Cyclist Volume Model for Greater Adelaide

This notebook sets up the graph for Adelaide's cycle with all infrastructure data, cycle routes, mesh blocks, and destination zones.

Run this notebook __once__ to save the graph as a geopackage in `./graph_cache/graph_setup.graphml`, from there it can be retrieved by `./GRAPH_LOAD.ipynb`

Select `cyclist-volumes-adelaide` as the kernel for this notebook.

__*Generate Adelaide Graph Network*__

In [ ]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd

ox.settings.log_console = True
ox.settings.use_cache = True
ox.settings.useful_tags_way += [
    "cycleway", "cycleway:left", "cycleway:right", "cycleway:both",
    "cycleway:left:lane", "cycleway:right:lane", "cycleway:both:lane",
    "cycleway:left:buffer:left", "cycleway:left:buffer:right", "cycleway:left:buffer:both",
    "cycleway:right:buffer:left", "cycleway:right:buffer:right", "cycleway:right:buffer:both",
    "cycleway:both:buffer:left", "cycleway:both:buffer:right", "cycleway:both:buffer:both",
    "cycleway:left:conditional", "cycleway:right:conditional", "cycleway:both:conditional",
    "cycleway:left:lane:conditional", "cycleway:right:lane:conditional", "cycleway:both:lane:conditional",
    "foot", "bicycle", "route", "name"
]

# Monkey-patch deprecated aliases so old code doesn't crash
import osmnx.settings as ox_settings
if not hasattr(ox_settings, 'http_user_agent'):
    ox_settings.http_user_agent = ox_settings.default_user_agent
if not hasattr(ox_settings, 'http_referer'):
    ox_settings.http_referer = ox_settings.default_referer
if not hasattr(ox_settings, 'nominatim_url'):
    ox_settings.nominatim_url = ox_settings.nominatim_endpoint
if not hasattr(ox_settings, 'http_accept_language'):
    ox_settings.http_accept_language = ox_settings.default_accept_language

In [ ]:
gdf_boundary = ox.geocode_to_gdf("R11381689", by_osmid=True)
polygon = gdf_boundary.geometry.iloc[0]

# Monash dual-filter approach to capture all relevant OSM tags
bike_filter_1 = '[~"highway"~"cycleway|trunk|primary|secondary|tertiary|unclassified|residential|primary_link|secondary_link|tertiary_link|living_street|trailhead|service"]["access"!~"no|private"]["bicycle"!~"no"]["area"!~"yes"]'
G1 = ox.graph_from_polygon(polygon, custom_filter=bike_filter_1, simplify=False, retain_all=True)

bike_filter_2 = '[~"highway"~"footway|pedestrian|path"]["bicycle"~"yes|designated|dismount"]["area"!~"yes"]'
G2 = ox.graph_from_polygon(polygon, custom_filter=bike_filter_2, simplify=False, retain_all=True)

G = nx.compose(G1, G2)

In [ ]:
# Debug: Inspect edge attributes loaded from OSM
import pandas as pd

nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

print(f"=== Graph Stats ===")
print(f"Nodes: {len(nodes)}")
print(f"Edges: {len(edges)}")
print(f"Edge columns ({len(edges.columns)}): {list(edges.columns)}")

print(f"\n=== Sample of cycleway-related tags ===")
cycleway_cols = [c for c in edges.columns if 'cycleway' in c.lower() or c in ['highway', 'foot', 'bicycle', 'name', 'route']]
print(f"Relevant columns: {cycleway_cols}")

for col in cycleway_cols:
    if col in edges.columns:
        non_null = edges[col].notna().sum()
        if non_null > 0:
            print(f"\n{col}: {non_null} non-null values")
            print(edges[col].value_counts().head(10))

print(f"\n=== First 5 edges ===")
print(edges[cycleway_cols].head())

# Check for key classification tags
key_tags = [
    'cycleway:left:buffer:left', 'cycleway:left:buffer:right', 'cycleway:left:buffer:both',
    'cycleway:both:buffer:left', 'cycleway:both:buffer:right', 'cycleway:both:buffer:both',
    'cycleway:left:lane', 'cycleway:both:lane',
    'cycleway:left:conditional', 'cycleway:left:lane:conditional',
    'cycleway:both:conditional', 'cycleway:both:lane:conditional',
    'highway', 'foot', 'bicycle', 'name', 'route'
]
print(f"\n=== Key classification tag coverage ===")
for tag in key_tags:
    if tag in edges.columns:
        nn = edges[tag].notna().sum()
        print(f"  {tag}: {nn} non-null")
    else:
        print(f"  {tag}: MISSING")

__*Add in cycle route relations*__

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent  # notebook lives in graph_generation/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from graph_generation.cycle_routes import enrich_graph_with_cycle_routes

# Get the graph with IS_CYCLE_ROUTE edge attribute
G = enrich_graph_with_cycle_routes(G)

__*Load MBs and DZNs into graph nodes*__

In [ ]:
mbs_aus = gpd.read_file("../all-data/australia-mesh-blocks-raw/MB_2021_AUST_GDA2020.shp")
dzns_aus = gpd.read_file("../all-data/australia-destination-zones-raw/DZN_2021_AUST_GDA2020.shp")

mb_gdf_boundary_crs = gdf_boundary.to_crs(mbs_aus.crs)
dzn_gdf_boundary_crs = gdf_boundary.to_crs(dzns_aus.crs)
mbs = gpd.clip(mbs_aus, mb_gdf_boundary_crs)
dzns = gpd.clip(dzns_aus, dzn_gdf_boundary_crs)

print('MB all count: ', len(mbs_aus))
print('MB Adelaide count: ', len(mbs))
print('DZN all count: ', len(dzns_aus))
print('DZN Adelaide count: ', len(dzns))

In [ ]:
nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)[["geometry"]]
 
nodes_mbs = gpd.sjoin(
    nodes_gdf.to_crs(mbs.crs),
    mbs[["MB_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_mb_codes = nodes_mbs["MB_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_mb_codes, "MB_CODE21")

nodes_dzns = gpd.sjoin(
    nodes_gdf.to_crs(dzns.crs),
    dzns[["DZN_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_dzn_codes = nodes_dzns["DZN_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_dzn_codes, "DZN_CODE21")

print(f"Graph nodes: {len(G.nodes)}")
print(f"Nodes with MB_CODE21: {len(node_mb_codes)}")
print(f"Nodes with DZN_CODE21: {len(node_dzn_codes)}")

In [ ]:
import pandas as pd
nodes_to_remove = [
    node_id
    for node_id, node_data in G.nodes(data=True)
    if not (
        pd.notna(node_data.get("MB_CODE21"))
        or pd.notna(node_data.get("DZN_CODE21"))
    )
]

G.remove_nodes_from(nodes_to_remove)
print(f"Removed {len(nodes_to_remove)} nodes; remaining nodes: {len(G.nodes)}")
print(f"Remaining edges: {len(G.edges)}")

In [ ]:
print(G)

# Export graph

Save all graph data so it can be retrieved by `./MAIN.ipynb`

In [ ]:
import os
import osmnx as ox

output_dir = "./graph_cache"
os.makedirs(output_dir, exist_ok=True)

graphml_path = os.path.join(output_dir, "graph_setup.graphml")
ox.save_graphml(G, filepath=graphml_path)
print(f"Graph saved to {graphml_path}")